# Tutorial: Chapter 4.1 Coupling Geometry

This notebook is a focused tutorial for the Chapter 4.1 coupling-geometry experiments. It shows how endpoint coupling choices affect conditional flow matching trajectories on EB from time `1` to `2`, using standardized PC-20 for training and metrics while reserving PHATE coordinates for display-only figures.

Tutorial goals:
- Compare independent random-product coupling with PC-20 OT coupling on the same EB bridge.
- Audit Sinkhorn epsilon and minibatch-coupling sensitivity without changing the main defaults.
- Train and evaluate two rectified-flow rounds from the Exp 1 OT-CFM base model.
- Emit the same paper-facing figures and tables, then check them in a split-specific artifact manifest.

Run this notebook top to bottom from a fresh kernel. The paper-facing artifacts are written to `figures/ch04` and `outputs/ch04`; cache and progress files stay under `outputs/ch04/cache`.


## 0. Setup

This section makes the notebook independently runnable: imports, project-root discovery, device selection, random seeds, output directories, plotting style, cache helpers, and default run-size controls are all defined here.

The key defaults remain the chapter defaults: `DEFAULT_SEED = 42`, EB time `1 -> 2`, `TRAINING_STEPS = 1500`, `BATCH_SIZE = 256`, Sinkhorn epsilon `0.05`, and the fixed epsilon grid `[0.01, 0.02, 0.05, 0.1, 0.5]`. Environment variables only shorten runs for smoke testing or resume partial work; they do not change the paper artifact names.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_ch04")


In [ ]:
from pathlib import Path
import sys
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass
from typing import Callable, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display


In [ ]:
try:
    import torch
    from torch import nn
except Exception as exc:
    raise ImportError("Chapter 4 experiments require PyTorch.") from exc

try:
    import anndata as ad
except Exception:
    ad = None

from scipy import sparse
from scipy.sparse.csgraph import shortest_path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/home/xmabs/flow_matching_for_dynamic_biology/flow_matching_for_dynamic_biology")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import VelocityMLP as ProjectVelocityMLP, count_parameters
from src.losses import cfm_batch, cfm_loss_from_pairs
from src.train import train_cfm_steps
from src.sampling import euler_sample
from src.samplers import CouplingPairSampler as SrcCouplingPairSampler
from src.ot import independent_coupling, coupling_diagnostics
from src.metrics import (
    endpoint_metrics,
    fate_mass_error,
    coupling_l1_distance,
    normalized_cost_matrix,
    distribution_readout_metrics,
)
from src.representations import (
    fit_pca_state_space,
    pca_inverse_transform,
    program_index_dict,
    readout_program_scores_from_matrix,
    standardize_train_space,
)


In [ ]:
SEEDS = [42, 43, 44]
DEFAULT_SEED = 42
SOURCE_TIME = "1"
TARGET_TIME = "2"
TRAINING_STEPS = int(os.environ.get("CH04_TRAINING_STEPS", "1500"))
BATCH_SIZE = int(os.environ.get("CH04_BATCH_SIZE", "256"))
DEFAULT_NFE = int(os.environ.get("CH04_DEFAULT_NFE", "64"))
NFE_GRID = [2, 4, 8, 16, 32, 64]
SINKHORN_EPSILON = float(os.environ.get("CH04_SINKHORN_EPSILON", "0.05"))
EPSILON_GRID = [0.01, 0.02, 0.05, 0.1, 0.5]
BOOTSTRAP_REPEATS = int(os.environ.get("CH04_BOOTSTRAP_REPEATS", "50"))
EB_MAX_CELLS_PER_TIME = os.environ.get("CH04_EB_MAX_CELLS_PER_TIME", "")
EB_MAX_CELLS_PER_TIME = None if EB_MAX_CELLS_PER_TIME == "" else int(EB_MAX_CELLS_PER_TIME)
TOY_TRAINING_STEPS = int(os.environ.get("CH04_TOY_TRAINING_STEPS", str(TRAINING_STEPS)))
SMOKE_MODE = os.environ.get("CH04_SMOKE_MODE", "0") == "1"
if SMOKE_MODE:
    TRAINING_STEPS = min(TRAINING_STEPS, 20)
    TOY_TRAINING_STEPS = min(TOY_TRAINING_STEPS, 20)
    BATCH_SIZE = min(BATCH_SIZE, 64)
    DEFAULT_NFE = min(DEFAULT_NFE, 8)
    NFE_GRID = [2, 4, 8]
    BOOTSTRAP_REPEATS = min(BOOTSTRAP_REPEATS, 3)
    EB_MAX_CELLS_PER_TIME = 120 if EB_MAX_CELLS_PER_TIME is None else min(EB_MAX_CELLS_PER_TIME, 120)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
EB_PATH = DATA_DIR / "trajectorynet_eb" / "eb_velocity_v5.npz"
TOY_DIR = DATA_DIR / "toy_branching_snapshots"
TOY_CSV_PATH = TOY_DIR / "observed_2d_snapshots.csv"
TOY_H5AD_PATH = TOY_DIR / "branching_toy_pseudocounts.h5ad"

FIG_DIR = PROJECT_ROOT /  "figures" / "ch04"
OUT_DIR = PROJECT_ROOT /  "outputs" / "ch04"
if SMOKE_MODE:
    FIG_DIR = FIG_DIR / "smoke"
    OUT_DIR = OUT_DIR / "smoke"
CACHE_DIR = OUT_DIR / "cache"
for path in [FIG_DIR, OUT_DIR, CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = {
    "source": "#4C78A8",
    "target": "#F58518",
    "random": "#8E8E8E",
    "ot": "#008A7A",
    "reflow1": "#5369A6",
    "reflow2": "#B279A2",
    "rare": "#D95F02",
    "major": "#2C7FB8",
    "program": "#54A24B",
    "diagnostic": "#E45756",
}


In [ ]:
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Training steps: {TRAINING_STEPS}; batch size: {BATCH_SIZE}; default NFE: {DEFAULT_NFE}")
print(f"Smoke mode: {SMOKE_MODE}")


## 1. Shared Utilities

These cells collect the reusable machinery needed by every experiment section. They keep the tutorial self-contained by defining the local model wrapper, CFM training loop, Sinkhorn coupling helpers, rollout metrics, PHATE projection helpers, plotting functions, and artifact save wrappers in one place.

No experiment is run in this section. Later cells reuse these utilities to write named figures, tables, JSON summaries, and cache files.


In [ ]:
from src.utils import set_seed as _set_seed


def set_seed(seed: int = DEFAULT_SEED) -> None:
    _set_seed(seed)


In [ ]:
from src.artifacts import json_ready, load_json, save_json


In [ ]:
from src.artifacts import load_npz, load_pt as _load_pt, save_csv, save_npz, save_pt


def load_pt(path: str | Path, map_location=None):
    return _load_pt(path, map_location=map_location or DEVICE)


In [ ]:
def save_figure(fig, filename: str, close: bool = True) -> Path:
    path = FIG_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    if close:
        plt.close(fig)
    return path


In [ ]:
from src.artifacts import artifact_exists, as_float32, ensure_finite, sample_rows as _sample_rows, stable_hash, to_tensor as _to_tensor


def sample_rows(n: int, max_n: int | None, seed: int = DEFAULT_SEED) -> np.ndarray:
    return _sample_rows(n, max_n, seed)


def to_tensor(x, device: torch.device = DEVICE):
    return _to_tensor(x, device)


In [ ]:
class VelocityMLP(ProjectVelocityMLP):
    """Chapter 4 VelocityMLP wrapper with requested defaults.

    The underlying implementation is imported from `src.models`, but the
    requested architecture is fixed here: hidden=128 and layers=4.
    """
    def __init__(self, input_dim: int, hidden: int = 128, layers: int = 4):
        super().__init__(x_dim=int(input_dim), hidden_dim=int(hidden), hidden_layers=int(layers))


from src.flow_runtime import make_time_batch as _make_time_batch


def make_time_batch(batch_size: int, device: torch.device = DEVICE) -> torch.Tensor:
    return _make_time_batch(batch_size, device)


In [ ]:
from src.ot import compute_cost_matrix, sample_from_plan as _sample_from_plan, sample_independent_pairs as _sample_independent_pairs, sinkhorn_plan as _sinkhorn_plan


def sample_independent_pairs(X0, X1, n_pairs: int, seed: int = DEFAULT_SEED):
    return _sample_independent_pairs(X0, X1, n_pairs=n_pairs, seed=seed)


def sinkhorn_plan(C, epsilon: float = SINKHORN_EPSILON, return_info: bool = False):
    return _sinkhorn_plan(C, epsilon=epsilon, return_info=return_info)


def sample_from_plan(pi, n_pairs: int, seed: int = DEFAULT_SEED):
    return _sample_from_plan(pi, n_pairs=n_pairs, seed=seed)


In [ ]:
class PlanPairSampler:
    """Notebook-local thin wrapper around `src.samplers.CouplingPairSampler`."""
    def __init__(self, X0, X1, pi=None, seed: int = DEFAULT_SEED, labels0=None, labels1=None):
        self.X0 = as_float32(X0)
        self.X1 = as_float32(X1)
        if pi is None:
            pi = independent_coupling(len(self.X0), len(self.X1))
        self.pi = np.asarray(pi, dtype=float)
        self.src_sampler = SrcCouplingPairSampler(self.X0, self.X1, self.pi, seed=seed, labels0=labels0, labels1=labels1)

    def sample(self, batch_size: int = BATCH_SIZE):
        return self.src_sampler.sample(int(batch_size))


In [ ]:
from src.flow_runtime import train_cfm as _train_cfm


def train_cfm(
    model,
    pair_sampler,
    steps: int = TRAINING_STEPS,
    batch_size: int = BATCH_SIZE,
    lr: float = 1e-3,
    device: torch.device = DEVICE,
    seed: int = DEFAULT_SEED,
    log_every: int = 250,
):
    kwargs = dict(steps=steps, batch_size=batch_size, lr=lr, device=device, seed=seed, log_every=log_every)
    return _train_cfm(model, pair_sampler, **kwargs)


In [ ]:
from src.flow_runtime import rollout_euler as _rollout_euler, trajectory_rollout as _trajectory_rollout


@torch.no_grad()
def rollout_euler(model, x0, nfe: int = DEFAULT_NFE, device: torch.device = DEVICE):
    return _rollout_euler(model, x0, nfe=nfe, device=device)


@torch.no_grad()
def trajectory_rollout(model, x0, nfe: int = DEFAULT_NFE, return_path: bool = True, device: torch.device = DEVICE):
    return _trajectory_rollout(model, x0, nfe=nfe, return_path=return_path, device=device)


In [ ]:
from src.metrics import mmd_rbf, path_energy, path_length, straightness, straightness_action_S, tortuosity_straightness
from src.metrics import sliced_w2 as _sliced_w2


def sliced_w2(X, Y, n_projections: int = 128, seed: int = DEFAULT_SEED):
    return _sliced_w2(X, Y, n_projections=n_projections, seed=seed)


In [ ]:
from src.metrics import off_manifold_knn

from src.flow_runtime import coarse_step_error as _coarse_step_error


def coarse_step_error(model, x0, nfe_coarse: int = 4, nfe_fine: int = 64):
    return _coarse_step_error(model, x0, nfe_coarse=nfe_coarse, nfe_fine=nfe_fine, device=DEVICE)


In [ ]:
from src.metrics import coupling_topk_overlap, effective_support, evaluate_endpoint as _evaluate_endpoint, plan_entropy, topk_nn_overlap


def evaluate_endpoint(pred, target, seed: int = DEFAULT_SEED):
    return _evaluate_endpoint(pred, target, seed=seed)


In [ ]:
def plot_phate_pairs(X0_phate, X1_phate, idx0, idx1, title: str, max_lines: int = 120, seed: int = DEFAULT_SEED, values=None):
    return plot_pair_panels(
        X0_phate,
        X1_phate,
        [{"title": title, "idx0": idx0, "idx1": idx1, "values": values, "seed": seed}],
        filename="_temporary_pair_panel.png",
        title=title,
    )


def plot_pair_panels(X0_phate, X1_phate, panels, filename: str, title: str, value_label: str = "PC-20 chord length"):
    from matplotlib.collections import LineCollection
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable

    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(5.0 * n, 4.2), squeeze=False)
    axes_flat = axes[0]
    all_values = []
    for panel in panels:
        if panel.get("values") is not None:
            all_values.append(np.asarray(panel["values"], dtype=float))
    norm = None
    if all_values:
        finite = np.concatenate([v[np.isfinite(v)] for v in all_values if np.any(np.isfinite(v))])
        norm = Normalize(vmin=float(finite.min()), vmax=float(finite.max())) if finite.size else None
    for ax, panel in zip(axes_flat, panels):
        idx0, idx1 = panel["idx0"], panel["idx1"]
        ax.scatter(X0_phate[:, 0], X0_phate[:, 1], s=8, color=PALETTE["source"], alpha=0.20, linewidths=0)
        ax.scatter(X1_phate[:, 0], X1_phate[:, 1], s=8, color=PALETTE["target"], alpha=0.20, linewidths=0)
        keep = sample_rows(len(idx0), min(panel.get("max_lines", 100), len(idx0)), seed=panel.get("seed", DEFAULT_SEED))
        segments = np.stack([X0_phate[idx0[keep]], X1_phate[idx1[keep]]], axis=1)
        values = panel.get("values")
        if values is not None and norm is not None:
            lc = LineCollection(segments, cmap="viridis", norm=norm, linewidths=0.8, alpha=0.55)
            lc.set_array(np.asarray(values, dtype=float)[keep])
            ax.add_collection(lc)
        else:
            lc = LineCollection(segments, colors=panel.get("color", "0.45"), linewidths=0.7, alpha=0.25)
            ax.add_collection(lc)
        ax.set_title(panel["title"])
        ax.set_xlabel("PHATE 1")
        ax.set_ylabel("PHATE 2")
    if norm is not None:
        fig.colorbar(ScalarMappable(norm=norm, cmap="viridis"), ax=list(axes_flat), fraction=0.035, pad=0.02, label=value_label)
    fig.suptitle(title, y=1.02)
    return save_figure(fig, filename)


In [ ]:
def add_local_arrows(ax, projected_traj, observed_phate, color, max_arrows: int = 28, seed: int = DEFAULT_SEED):
    """Draw local average arrows only for trajectory points close to observed PHATE neighborhoods."""
    projected_traj = np.asarray(projected_traj, dtype=float)
    observed_phate = np.asarray(observed_phate, dtype=float)
    if projected_traj.shape[0] < 3 or projected_traj.shape[-1] != 2:
        return
    mid_step = projected_traj.shape[0] // 2
    anchors = projected_traj[mid_step]
    deltas = projected_traj[min(mid_step + 1, projected_traj.shape[0] - 1)] - projected_traj[max(mid_step - 1, 0)]
    nn_obs = NearestNeighbors(n_neighbors=min(15, len(observed_phate))).fit(observed_phate)
    obs_r = nn_obs.kneighbors(observed_phate, return_distance=True)[0][:, -1]
    dist = nn_obs.kneighbors(anchors, return_distance=True)[0][:, 0]
    threshold = float(np.quantile(obs_r, 0.75))
    valid = np.flatnonzero(dist <= threshold)
    if valid.size == 0:
        return
    keep = sample_rows(len(valid), min(max_arrows, len(valid)), seed=seed)
    valid = valid[keep]
    ax.quiver(
        anchors[valid, 0], anchors[valid, 1], deltas[valid, 0], deltas[valid, 1],
        angles="xy", scale_units="xy", scale=1.0, width=0.004, color=color, alpha=0.75,
    )


In [ ]:
def plot_projected_trajectories(paths, X0_phate, X1_phate, pc_to_phate, filename: str, title: str, max_lines: int = 45, local_arrows: bool = True):
    n = len(paths)
    fig, axes = plt.subplots(1, n, figsize=(5.0 * n, 4.2), squeeze=False)
    observed_phate = np.vstack([X0_phate, X1_phate])
    for ax, (name, traj) in zip(axes[0], paths.items()):
        ax.scatter(X0_phate[:, 0], X0_phate[:, 1], s=8, color=PALETTE["source"], alpha=0.18, linewidths=0)
        ax.scatter(X1_phate[:, 0], X1_phate[:, 1], s=8, color=PALETTE["target"], alpha=0.16, linewidths=0)
        keep = sample_rows(traj.shape[1], min(max_lines, traj.shape[1]), seed=DEFAULT_SEED)
        selected = np.asarray(traj[:, keep, :], dtype=np.float32)
        flat = selected.reshape(-1, selected.shape[-1])
        ph = pc_to_phate(flat).reshape(selected.shape[0], selected.shape[1], 2)
        color = PALETTE.get(name, "0.35")
        for j in range(ph.shape[1]):
            ax.plot(ph[:, j, 0], ph[:, j, 1], color=color, alpha=0.55, linewidth=1.0)
        if local_arrows:
            add_local_arrows(ax, ph, observed_phate, color=color, seed=DEFAULT_SEED + 7)
        ax.set_title(name.replace("_", " "))
        ax.set_xlabel("PHATE 1 (display only)")
        ax.set_ylabel("PHATE 2 (display only)")
    fig.suptitle(title, y=1.02)
    return save_figure(fig, filename)


In [ ]:
def plot_metric_lines(table, x: str, y: str, hue: str, filename: str, title: str):
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    for name, group in table.groupby(hue):
        group = group.sort_values(x)
        ax.plot(group[x], group[y], marker="o", linewidth=1.5, label=str(name).replace("_", " "))
    ax.set_xscale("log", base=2 if set(table[x]).issubset(set(NFE_GRID)) else 10)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)
    ax.legend(frameon=False)
    return save_figure(fig, filename)


def plot_heatmap(matrix, title: str, filename: str, max_size: int = 120, cmap: str = "viridis"):
    M = np.asarray(matrix)
    rows = sample_rows(M.shape[0], min(max_size, M.shape[0]), seed=DEFAULT_SEED)
    cols = sample_rows(M.shape[1], min(max_size, M.shape[1]), seed=DEFAULT_SEED + 1)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(M[np.ix_(rows, cols)], aspect="auto", cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel("target subset")
    ax.set_ylabel("source subset")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return save_figure(fig, filename)


def plot_table_image(frame: pd.DataFrame, filename: str, title: str, max_rows: int = 12):
    shown = frame.head(max_rows).copy()
    fig, ax = plt.subplots(figsize=(min(14, 1.5 + 1.4 * shown.shape[1]), 0.8 + 0.35 * len(shown)))
    ax.axis("off")
    ax.set_title(title, loc="left")
    tbl = ax.table(cellText=shown.round(4).astype(str).values, colLabels=shown.columns, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1.0, 1.25)
    return save_figure(fig, filename)


In [ ]:
def display_figure(filename: str, width: int = 900) -> Path:
    path = FIG_DIR / filename
    if not path.exists():
        raise FileNotFoundError(path)
    display(Image(filename=str(path), width=width))
    return path


### Small-figure plotting helpers
These helpers keep each output figure focused on one claim while reusing the same styling, PHATE display limits, deterministic representative selection, and PC-20 metric annotations.

In [ ]:

METHOD_LABELS = {
    "random_cfm": "Random CFM",
    "ot_cfm": "OT-CFM",
    "reflow_1": "Reflow 1",
    "reflow_2": "Reflow 2",
    "random": "Random CFM",
    "ot": "OT-CFM",
    "reflow1": "Reflow 1",
    "reflow2": "Reflow 2",
}
METHOD_COLORS = {
    "random_cfm": PALETTE["random"],
    "ot_cfm": PALETTE["ot"],
    "reflow_1": PALETTE["reflow1"],
    "reflow_2": PALETTE["reflow2"],
    "random": PALETTE["random"],
    "ot": PALETTE["ot"],
    "reflow1": PALETTE["reflow1"],
    "reflow2": PALETTE["reflow2"],
}
REP_PAIR_QUANTILES = (0.50, 0.75, 0.90, 0.95)
REP_TRAJ_QUANTILES = (0.35, 0.55, 0.75, 0.90)


def method_label(method: str) -> str:
    return METHOD_LABELS.get(str(method), str(method).replace("_", " "))


def method_color(method: str) -> str:
    return METHOD_COLORS.get(str(method), "0.35")


def phate_limits(*arrays, pad_fraction: float = 0.055):
    pts = np.vstack([np.asarray(a, dtype=float)[:, :2] for a in arrays])
    lo = pts.min(axis=0)
    hi = pts.max(axis=0)
    span = np.maximum(hi - lo, 1e-6)
    pad = span * float(pad_fraction)
    return (lo[0] - pad[0], hi[0] + pad[0]), (lo[1] - pad[1], hi[1] + pad[1])


def apply_phate_axes(ax, limits, xlabel: bool = True, ylabel: bool = True):
    ax.set_xlim(*limits[0])
    ax.set_ylim(*limits[1])
    ax.set_aspect("equal", adjustable="box")
    if xlabel:
        ax.set_xlabel("PHATE 1 (display only)")
    else:
        ax.set_xlabel("")
    if ylabel:
        ax.set_ylabel("PHATE 2 (display only)")
    else:
        ax.set_ylabel("")


def draw_eb_background(ax, X0_phate, X1_phate, limits=None, source_alpha: float = 0.10, target_alpha: float = 0.10):
    ax.scatter(X0_phate[:, 0], X0_phate[:, 1], s=8, color=PALETTE["source"], alpha=source_alpha, linewidths=0)
    ax.scatter(X1_phate[:, 0], X1_phate[:, 1], s=8, color=PALETTE["target"], alpha=target_alpha, linewidths=0)
    if limits is not None:
        apply_phate_axes(ax, limits)


def select_representatives_by_quantile(values, quantiles=REP_PAIR_QUANTILES):
    values = np.asarray(values, dtype=float)
    finite = np.flatnonzero(np.isfinite(values))
    if finite.size == 0:
        raise ValueError("No finite values available for representative selection")
    used = set()
    rows = []
    for q in quantiles:
        target = float(np.quantile(values[finite], q))
        order = finite[np.argsort(np.abs(values[finite] - target), kind="mergesort")]
        pick = next((int(i) for i in order if int(i) not in used), int(order[0]))
        used.add(pick)
        rows.append({"row_index": pick, "quantile": float(q), "value": float(values[pick]), "target": target})
    return pd.DataFrame(rows)


def selected_pair_frame(pair_frame, quantiles=REP_PAIR_QUANTILES):
    reps = select_representatives_by_quantile(pair_frame["pc20_chord_length"].to_numpy(), quantiles=quantiles)
    selected = pair_frame.iloc[reps["row_index"].to_numpy()].copy().reset_index(drop=True)
    selected["selected_quantile"] = reps["quantile"].to_numpy()
    selected["selection_target_pc20_chord_length"] = reps["target"].to_numpy()
    return selected


def plot_representative_pairs(pair_frame, X0_phate, X1_phate, filename: str, title: str, color: str, limits):
    selected = selected_pair_frame(pair_frame)
    fig, ax = plt.subplots(figsize=(4.35, 4.05))
    draw_eb_background(ax, X0_phate, X1_phate, limits=limits, source_alpha=0.10, target_alpha=0.10)
    for _, row in selected.iterrows():
        i0 = int(row["idx0"])
        i1 = int(row["idx1"])
        p0 = X0_phate[i0]
        p1 = X1_phate[i1]
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color=color, linewidth=1.9, alpha=0.88, solid_capstyle="round")
    idx0 = selected["idx0"].astype(int).to_numpy()
    idx1 = selected["idx1"].astype(int).to_numpy()
    ax.scatter(X0_phate[idx0, 0], X0_phate[idx0, 1], s=58, color=PALETTE["source"], edgecolor="white", linewidth=0.8, zorder=4, label="selected source")
    ax.scatter(X1_phate[idx1, 0], X1_phate[idx1, 1], s=72, marker="^", color=PALETTE["target"], edgecolor="white", linewidth=0.8, zorder=5, label="selected target")
    ax.plot([], [], color=color, linewidth=1.9, label="endpoint chord")
    ax.set_title(title)
    ax.legend(loc="upper right", frameon=False, handlelength=1.6, borderpad=0.2)
    path = save_figure(fig, filename)
    display_figure(filename, width=620)
    display(selected[["idx0", "idx1", "pc20_chord_length", "selected_quantile"]].round(4))
    return path, selected


def plot_chord_length_ecdf(pair_stats_by_method, metrics_table, filename: str):
    fig, ax = plt.subplots(figsize=(4.6, 3.75))
    for method in ["random_cfm", "ot_cfm"]:
        vals = np.asarray(pair_stats_by_method[method]["pc20_chord_length"], dtype=float)
        vals = np.sort(vals[np.isfinite(vals)])
        y = np.arange(1, vals.size + 1) / float(vals.size)
        color = method_color(method)
        row = metrics_table.loc[metrics_table["method"] == method].iloc[0]
        mean = float(row["mean_pc20_chord_length"])
        median = float(row["median_pc20_chord_length"])
        ax.plot(vals, y, color=color, linewidth=2.0, label=f"{method_label(method)} mean {mean:.2f}; median {median:.2f}")
        ax.axvline(median, color=color, linestyle="--", linewidth=1.0, alpha=0.55)
    ax.set_title("Endpoint chord length distribution")
    ax.set_xlabel("Endpoint chord length in standardized PC-20")
    ax.set_ylabel("ECDF")
    ax.set_ylim(0, 1.02)
    ax.legend(loc="lower right", frameon=False)
    path = save_figure(fig, filename)
    display_figure(filename, width=650)
    display(metrics_table[["method", "mean_pc20_chord_length", "median_pc20_chord_length", "std_pc20_chord_length", "training_space"]].round(4))
    return path


def rollout_representative_indices(reference_traj, quantiles=REP_TRAJ_QUANTILES):
    displacement = np.linalg.norm(np.asarray(reference_traj[-1] - reference_traj[0], dtype=float), axis=1)
    reps = select_representatives_by_quantile(displacement, quantiles=quantiles)
    return reps["row_index"].astype(int).to_numpy(), reps


def project_selected_trajectory(traj, selected_idx, pc_to_phate_fn):
    selected = np.asarray(traj[:, selected_idx, :], dtype=np.float32)
    flat = selected.reshape(-1, selected.shape[-1])
    return pc_to_phate_fn(flat).reshape(selected.shape[0], selected.shape[1], 2)


def plot_representative_rollouts(traj, selected_idx, X0_phate, X1_phate, pc_to_phate_fn, filename: str, title: str, color: str, limits):
    ph = project_selected_trajectory(traj, selected_idx, pc_to_phate_fn)
    fig, ax = plt.subplots(figsize=(4.35, 4.05))
    draw_eb_background(ax, X0_phate, X1_phate, limits=limits, source_alpha=0.09, target_alpha=0.09)
    for j in range(ph.shape[1]):
        ax.plot(ph[:, j, 0], ph[:, j, 1], color=color, alpha=0.86, linewidth=2.0, solid_capstyle="round")
        start = max(0, ph.shape[0] - 6)
        ax.annotate("", xy=ph[-1, j], xytext=ph[start, j], arrowprops={"arrowstyle": "->", "color": color, "lw": 1.6, "shrinkA": 0, "shrinkB": 0, "mutation_scale": 10})
    ax.scatter(ph[0, :, 0], ph[0, :, 1], s=58, color=PALETTE["source"], edgecolor="white", linewidth=0.8, zorder=4, label="source")
    ax.scatter(ph[-1, :, 0], ph[-1, :, 1], s=70, marker="X", color=color, edgecolor="white", linewidth=0.8, zorder=5, label="generated endpoint")
    ax.scatter([], [], s=28, color=PALETTE["target"], alpha=0.55, linewidths=0, label="EB target background")
    ax.set_title(title)
    ax.legend(loc="upper right", frameon=False, handlelength=1.5, borderpad=0.2)
    path = save_figure(fig, filename)
    display_figure(filename, width=620)
    return path


def observed_near_velocity_indices(reference_traj, pc_to_phate_fn, observed_phate, n_arrows: int = 6, seed: int = DEFAULT_SEED):
    mid = reference_traj.shape[0] // 2
    anchors = pc_to_phate_fn(reference_traj[mid].astype(np.float32))
    nn_obs = NearestNeighbors(n_neighbors=min(15, len(observed_phate))).fit(observed_phate)
    obs_r = nn_obs.kneighbors(observed_phate, return_distance=True)[0][:, -1]
    dist = nn_obs.kneighbors(anchors, return_distance=True)[0][:, 0]
    valid = np.flatnonzero(dist <= float(np.quantile(obs_r, 0.85)))
    if valid.size == 0:
        valid = np.arange(reference_traj.shape[1])
    disp = np.linalg.norm(reference_traj[-1] - reference_traj[0], axis=1)
    valid_disp = disp[valid]
    reps = select_representatives_by_quantile(valid_disp, quantiles=np.linspace(0.25, 0.90, int(n_arrows)))
    return valid[reps["row_index"].astype(int).to_numpy()]


def plot_local_velocity_examples(trajs, selected_idx, X0_phate, X1_phate, pc_to_phate_fn, filename: str, limits):
    methods = [("random_cfm", trajs["random_cfm"]), ("ot_cfm", trajs["ot_cfm"])]
    fig, axes = plt.subplots(1, 2, figsize=(8.1, 3.85), sharex=True, sharey=True)
    for ax, (method, traj) in zip(axes, methods):
        color = method_color(method)
        draw_eb_background(ax, X0_phate, X1_phate, limits=limits, source_alpha=0.08, target_alpha=0.08)
        mid = traj.shape[0] // 2
        step = min(mid + 3, traj.shape[0] - 1)
        anchors = pc_to_phate_fn(traj[mid, selected_idx, :].astype(np.float32))
        next_points = pc_to_phate_fn(traj[step, selected_idx, :].astype(np.float32))
        delta = next_points - anchors
        scale = 2.0
        ax.quiver(anchors[:, 0], anchors[:, 1], delta[:, 0] * scale, delta[:, 1] * scale, angles="xy", scale_units="xy", scale=1.0, width=0.0065, color=color, alpha=0.9)
        ax.scatter(anchors[:, 0], anchors[:, 1], s=40, color=color, edgecolor="white", linewidth=0.7, zorder=4)
        ax.set_title(f"{method_label(method)} local arrows")
    axes[0].set_ylabel("PHATE 2 (display only)")
    axes[1].set_ylabel("")
    path = save_figure(fig, filename)
    display_figure(filename, width=850)
    return path


def plot_reflow_representative_trajectories(trajs, selected_idx, X0_phate, X1_phate, pc_to_phate_fn, filename: str, limits):
    methods = [("ot_cfm", "OT-CFM"), ("reflow_1", "Reflow 1"), ("reflow_2", "Reflow 2")]
    fig, axes = plt.subplots(1, 3, figsize=(11.4, 3.85), sharex=True, sharey=True)
    for ax, (method, title) in zip(axes, methods):
        color = method_color(method)
        draw_eb_background(ax, X0_phate, X1_phate, limits=limits, source_alpha=0.075, target_alpha=0.075)
        ph = project_selected_trajectory(trajs[method], selected_idx, pc_to_phate_fn)
        for j in range(ph.shape[1]):
            ax.plot(ph[:, j, 0], ph[:, j, 1], color=color, linewidth=1.9, alpha=0.86, solid_capstyle="round")
            start = max(0, ph.shape[0] - 6)
            ax.annotate("", xy=ph[-1, j], xytext=ph[start, j], arrowprops={"arrowstyle": "->", "color": color, "lw": 1.45, "shrinkA": 0, "shrinkB": 0, "mutation_scale": 9})
        ax.scatter(ph[0, :, 0], ph[0, :, 1], s=45, color=PALETTE["source"], edgecolor="white", linewidth=0.7, zorder=4)
        ax.scatter(ph[-1, :, 0], ph[-1, :, 1], s=58, marker="X", color=color, edgecolor="white", linewidth=0.7, zorder=5)
        ax.set_title(title)
    for ax in axes[1:]:
        ax.set_ylabel("")
    path = save_figure(fig, filename)
    display_figure(filename, width=980)
    return path


def plot_metric_bar_grid(table, methods, metric_specs, filename: str, title: str | None = None, log_metrics=None):
    log_metrics = set(log_metrics or [])
    plot_table = table.set_index("method").loc[list(methods)].reset_index()
    n_metrics = len(metric_specs)
    ncols = min(4, n_metrics)
    nrows = int(math.ceil(n_metrics / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.1 * ncols, 3.0 * nrows), squeeze=False)
    for ax, (metric, label, ylabel) in zip(axes.ravel(), metric_specs):
        vals = plot_table[metric].astype(float).to_numpy()
        colors = [method_color(m) for m in plot_table["method"]]
        x = np.arange(len(vals))
        ax.bar(x, vals, color=colors, alpha=0.88, width=0.68)
        ax.set_xticks(x)
        ax.set_xticklabels([method_label(m) for m in plot_table["method"]], rotation=25, ha="right")
        ax.set_title(label)
        ax.set_ylabel(ylabel)
        if metric in log_metrics:
            ax.set_yscale("log")
        finite_vals = vals[np.isfinite(vals) & (vals > 0 if metric in log_metrics else np.isfinite(vals))]
        if finite_vals.size:
            ymax = float(np.nanmax(vals))
            ymin = float(np.nanmin(vals))
            if metric in log_metrics:
                ax.set_ylim(max(min(finite_vals) * 0.55, 1e-5), ymax * 1.8)
            else:
                ax.set_ylim(0, ymax * 1.18 if ymax > 0 else 1.0)
        for xi, yi in zip(x, vals):
            if not np.isfinite(yi):
                continue
            text_y = yi * (1.10 if metric in log_metrics else 1.025)
            ax.text(xi, text_y, f"{yi:.3g}", ha="center", va="bottom", fontsize=7)
    for ax in axes.ravel()[len(metric_specs):]:
        ax.axis("off")
    if title:
        fig.suptitle(title, y=1.02)
    path = save_figure(fig, filename)
    display_figure(filename, width=950)
    display(plot_table[["method"] + [m[0] for m in metric_specs]].round(4))
    return path


## 2. Load EB Data

This data section loads `data/trajectorynet_eb/eb_velocity_v5.npz`, selects the EB bridge from time `1` to time `2`, takes the first 20 PCs for training and endpoint metrics, and standardizes PC space using all EB snapshots. PHATE coordinates are retained only for visualization.

Main outputs from this section:
- `outputs/ch04/eb_data_summary.json` records the loaded EB split and preprocessing metadata.
- standardized PC endpoint arrays and the PC-to-PHATE display mapping are prepared for the experiment sections.

In [ ]:
def load_eb_data(path: Path = EB_PATH, source_time: str = SOURCE_TIME, target_time: str = TARGET_TIME):
    if not path.exists():
        raise FileNotFoundError(path)
    z = np.load(path, allow_pickle=True)
    keys = list(z.files)
    pcs_full = np.asarray(z["pcs"], dtype=np.float32)
    phate = np.asarray(z["phate"], dtype=np.float32)[:, :2]
    labels = np.asarray(z["sample_labels"]).astype(str)
    pcs20_raw = pcs_full[:, :20].astype(np.float32)
    mean = pcs20_raw.mean(axis=0)
    std = pcs20_raw.std(axis=0)
    std = np.where(std < 1e-6, 1.0, std)
    pcs20 = ((pcs20_raw - mean) / std).astype(np.float32)
    available = sorted(np.unique(labels).tolist(), key=lambda s: int(s) if str(s).isdigit() else str(s))
    if str(source_time) not in available or str(target_time) not in available:
        raise ValueError(f"Requested bridge {source_time}->{target_time}; available labels: {available}")
    idx_source_all = np.flatnonzero(labels == str(source_time))
    idx_target_all = np.flatnonzero(labels == str(target_time))
    if EB_MAX_CELLS_PER_TIME is not None:
        idx_source = idx_source_all[sample_rows(len(idx_source_all), EB_MAX_CELLS_PER_TIME, seed=DEFAULT_SEED)]
        idx_target = idx_target_all[sample_rows(len(idx_target_all), EB_MAX_CELLS_PER_TIME, seed=DEFAULT_SEED + 1)]
    else:
        idx_source, idx_target = idx_source_all, idx_target_all
    counts = pd.Series(labels, name="time").value_counts().sort_index().rename_axis("time").reset_index(name="n_cells")
    summary = {
        "source_path": str(path),
        "available_keys": keys,
        "pcs_shape_actual": list(pcs_full.shape),
        "pc20_shape_used": list(pcs20.shape),
        "phate_shape": list(phate.shape),
        "sample_label_values": counts.to_dict(orient="records"),
        "source_time": str(source_time),
        "target_time": str(target_time),
        "n_source_full": int(len(idx_source_all)),
        "n_target_full": int(len(idx_target_all)),
        "n_source_used": int(len(idx_source)),
        "n_target_used": int(len(idx_target)),
        "training_space": "standardized PC-20 from pcs[:, :20]",
        "ot_cost_space": "standardized PC-20 unless Exp 7 contrastive diagnostic",
        "display_space": "PHATE 2D only for visualization",
        "distributional_metrics_space": "standardized PC-20",
        "standardization": "mean/std fit on all EB snapshots in PC-20",
        "adaptation_note": "Input pcs has 100 columns; this chapter uses the first 20 PCs as PC-20.",
    }
    save_json(OUT_DIR / "eb_data_summary.json", summary)
    return {
        "keys": keys,
        "pcs20_all": pcs20,
        "pcs20_raw_all": pcs20_raw,
        "phate_all": phate,
        "labels": labels,
        "counts": counts,
        "pc_mean": mean,
        "pc_std": std,
        "idx_source": idx_source,
        "idx_target": idx_target,
        "X0_pc": pcs20[idx_source],
        "X1_pc": pcs20[idx_target],
        "X0_phate": phate[idx_source],
        "X1_phate": phate[idx_target],
        "source_time": str(source_time),
        "target_time": str(target_time),
        "summary": summary,
    }


In [ ]:
EB = load_eb_data()
print("EB keys:", EB["keys"])
print(EB["counts"])
print("Source/target PC shapes:", EB["X0_pc"].shape, EB["X1_pc"].shape)
print("Summary saved to", OUT_DIR / "eb_data_summary.json")


In [ ]:
# Display-only mapping from PC-20 trajectory points back to PHATE for plotting.
# This is not used for training or endpoint distributional metrics.
pc_to_phate_knn = KNeighborsClassifier(n_neighbors=min(15, len(EB["pcs20_all"])), weights="distance")
pc_to_phate_knn.fit(EB["pcs20_all"], np.arange(len(EB["pcs20_all"])))

def pc_to_phate(points_pc):
    points_pc = np.asarray(points_pc, dtype=np.float32)
    dist, ind = pc_to_phate_knn.kneighbors(points_pc, return_distance=True)
    w = 1.0 / np.clip(dist, 1e-6, None)
    w = w / w.sum(axis=1, keepdims=True)
    return np.einsum("nk,nkd->nd", w, EB["phate_all"][ind])

off_manifold_reference_pc = EB["pcs20_all"]
off_manifold_reference_note = "all available EB snapshots in standardized PC-20"
print(off_manifold_reference_note, off_manifold_reference_pc.shape)


## Exp 1. Independent vs OT Coupling on EB

This experiment compares two endpoint-coupling choices on the same EB source and target snapshots: a random product coupling and a PC-20 OT coupling. Both train `VelocityMLP(input_dim, hidden=128, layers=4)` with Adam lr `1e-3`, the chapter training-step default, and batch size 256.

Paper-facing outputs from Exp 1:
- `outputs/ch04/exp1_metrics.csv`
- `figures/ch04/fig4_1_independent_coupling_paths.png`
- `figures/ch04/fig4_2_random_vs_ot_pairs.png`
- `figures/ch04/fig4_1_supp_pair_statistics.png`
- `figures/ch04/fig4_5_random_vs_ot_projected_trajectories.png`

Cache outputs include the sampled couplings, Sinkhorn info, training histories, rollouts, and pair-level statistics under `outputs/ch04/cache`.


### 1A. Construct the EB coupling plans
The cost matrix, Sinkhorn plan, and independent product plan are created explicitly before any model training. These are the pair topologies every later metric refers back to.


In [ ]:
X0_eb, X1_eb = EB["X0_pc"], EB["X1_pc"]
X0p_eb, X1p_eb = EB["X0_phate"], EB["X1_phate"]
coupling_cache = CACHE_DIR / "exp1_eb_couplings.npz"
if coupling_cache.exists():
    cached_couplings = load_npz(coupling_cache)
    if cached_couplings["pi_ot"].shape == (len(X0_eb), len(X1_eb)) and cached_couplings["pi_random"].shape == (len(X0_eb), len(X1_eb)):
        C_eb_norm = cached_couplings["C_eb_norm"]
        pi_eb_ot = cached_couplings["pi_ot"]
        pi_eb_random = cached_couplings["pi_random"]
        C_eb_scale = float(np.asarray(cached_couplings["cost_scale"]))
        info_eb_ot = load_json(CACHE_DIR / "exp1_sinkhorn_info.json") if (CACHE_DIR / "exp1_sinkhorn_info.json").exists() else {"source": "cached exp1_eb_couplings.npz"}
    else:
        C_eb_norm, C_eb_scale = compute_cost_matrix(X0_eb, X1_eb, normalize=True)
        pi_eb_ot, info_eb_ot = sinkhorn_plan(C_eb_norm, epsilon=SINKHORN_EPSILON, return_info=True)
        pi_eb_random = independent_coupling(len(X0_eb), len(X1_eb))
else:
    C_eb_norm, C_eb_scale = compute_cost_matrix(X0_eb, X1_eb, normalize=True)
    pi_eb_ot, info_eb_ot = sinkhorn_plan(C_eb_norm, epsilon=SINKHORN_EPSILON, return_info=True)
    pi_eb_random = independent_coupling(len(X0_eb), len(X1_eb))
save_npz(coupling_cache, C_eb_norm=C_eb_norm, pi_ot=pi_eb_ot, pi_random=pi_eb_random, cost_scale=C_eb_scale)
save_json(CACHE_DIR / "exp1_sinkhorn_info.json", info_eb_ot)
print("OT Sinkhorn info:", info_eb_ot)


### 1B. Train or load the two CFM bridges
The cache path encodes input dimension, training steps, batch size, and seed. If a matching checkpoint exists, the notebook loads it; otherwise it trains the model and writes the history.


In [ ]:
def train_or_load_model(name: str, X0, X1, pi, steps: int = TRAINING_STEPS, seed: int = DEFAULT_SEED):
    cache_tag = f"d{X0.shape[1]}_steps{int(steps)}_batch{BATCH_SIZE}_seed{int(seed)}"
    ckpt = CACHE_DIR / f"{name}_{cache_tag}_model.pt"
    hist_path = CACHE_DIR / f"{name}_{cache_tag}_history.csv"
    model = VelocityMLP(input_dim=X0.shape[1], hidden=128, layers=4).to(DEVICE)
    if ckpt.exists():
        payload = load_pt(ckpt, map_location=DEVICE)
        if int(payload.get("input_dim", -1)) != int(X0.shape[1]) or int(payload.get("steps", -1)) != int(steps):
            raise ValueError(f"Checkpoint metadata mismatch for {ckpt}")
        model.load_state_dict(payload["state_dict"])
        history = pd.read_csv(hist_path) if hist_path.exists() else pd.DataFrame()
        print(f"Loaded {name} from {ckpt}")
        return model, history
    sampler = PlanPairSampler(X0, X1, pi=pi, seed=seed)
    history = train_cfm(model, sampler, steps=steps, batch_size=BATCH_SIZE, lr=1e-3, device=DEVICE, seed=seed)
    save_pt(ckpt, {"state_dict": model.state_dict(), "input_dim": int(X0.shape[1]), "seed": int(seed), "steps": int(steps), "batch_size": int(BATCH_SIZE)})
    save_csv(hist_path, history)
    print(f"Trained {name}; parameters={count_parameters(model)}; final loss={history.loss.iloc[-1]:.4f}")
    return model, history


In [ ]:
random_model, random_hist = train_or_load_model("exp1_random_cfm", X0_eb, X1_eb, pi_eb_random, seed=42)
ot_model, ot_hist = train_or_load_model("exp1_ot_cfm", X0_eb, X1_eb, pi_eb_ot, seed=42)


### 1C. Measure endpoint fit and pair geometry
The next cells keep the diagnostic construction visible: sample paths, compute endpoint metrics, audit local chord-direction dispersion, and save the Exp 1 table.


In [ ]:
def midpoint_direction_dispersion(X0, X1, pi, n_pairs: int = 4096, k: int = 25, seed: int = DEFAULT_SEED):
    """Neighborhood dispersion of sampled chord directions around pair midpoints."""
    idx0, idx1 = sample_from_plan(pi, n_pairs, seed=seed)
    x0, x1 = X0[idx0], X1[idx1]
    chords = x1 - x0
    chord_norm = np.linalg.norm(chords, axis=1)
    directions = chords / np.clip(chord_norm[:, None], 1e-12, None)
    midpoints = 0.5 * (x0 + x1)
    n_neighbors = max(2, min(int(k), len(midpoints)))
    nn = NearestNeighbors(n_neighbors=n_neighbors).fit(midpoints)
    neigh = nn.kneighbors(midpoints, return_distance=False)
    angular_std = []
    for ids in neigh:
        local_dirs = directions[ids]
        mean_dir = local_dirs.mean(axis=0)
        mean_norm = np.linalg.norm(mean_dir)
        if mean_norm < 1e-12:
            angles = np.full(len(ids), 90.0)
        else:
            mean_dir = mean_dir / mean_norm
            cosang = np.clip(local_dirs @ mean_dir, -1.0, 1.0)
            angles = np.degrees(np.arccos(cosang))
        angular_std.append(float(np.std(angles)))
    stats = pd.DataFrame({
        "idx0": idx0,
        "idx1": idx1,
        "pc20_chord_length": chord_norm,
        "midpoint_direction_angular_std_deg": angular_std,
    })
    return stats


In [ ]:
exp1_rows = []
exp1_paths = {}
pair_stats_exp1 = {}
for name, model, pi in [("random_cfm", random_model, pi_eb_random), ("ot_cfm", ot_model, pi_eb_ot)]:
    rollout_cache = CACHE_DIR / f"exp1_{name}_rollout.npz"
    if rollout_cache.exists():
        rollout_payload = load_npz(rollout_cache)
        z = rollout_payload["endpoint"]
        traj = rollout_payload["trajectory"]
        times = rollout_payload["times"]
    else:
        z, traj, times = trajectory_rollout(model, X0_eb, nfe=DEFAULT_NFE, return_path=True)
        save_npz(rollout_cache, endpoint=z, trajectory=traj, times=times)
    exp1_paths[name] = traj
    metrics = evaluate_endpoint(z, X1_eb)
    pair_stats = midpoint_direction_dispersion(X0_eb, X1_eb, pi, n_pairs=4096, k=25, seed=44)
    pair_stats["method"] = name
    pair_stats_exp1[name] = pair_stats
    chord = pair_stats["pc20_chord_length"].to_numpy()
    disp = pair_stats["midpoint_direction_angular_std_deg"].to_numpy()
    row = {
        "method": name,
        **metrics,
        "mean_pc20_chord_length": float(chord.mean()),
        "median_pc20_chord_length": float(np.median(chord)),
        "std_pc20_chord_length": float(chord.std()),
        "midpoint_direction_angular_std_mean_deg": float(disp.mean()),
        "midpoint_direction_angular_std_median_deg": float(np.median(disp)),
        "midpoint_direction_angular_std_iqr_deg": float(np.quantile(disp, 0.75) - np.quantile(disp, 0.25)),
        "training_space": "standardized PC-20",
        "cost_space": "standardized PC-20" if name == "ot_cfm" else "independent product coupling",
        "display_space": "PHATE 2D only for figures",
        "endpoint_metric_space": "standardized PC-20",
    }
    exp1_rows.append(row)


In [ ]:
exp1_pair_stats = pd.concat(pair_stats_exp1.values(), ignore_index=True)
save_csv(CACHE_DIR / "exp1_pair_level_statistics.csv", exp1_pair_stats)
exp1_metrics = pd.DataFrame(exp1_rows)
save_csv(OUT_DIR / "exp1_metrics.csv", exp1_metrics)
exp1_metrics


### 1D. Save and display Exp 1 figures
Each saved figure is displayed immediately so the notebook output matches the paper-facing artifacts written to disk.


### 1D. Random representative endpoint pairs
Claim: independent/random coupling is a legal endpoint sampler but creates long, crossing, direction-mixed chords. Data source: `pair_stats_exp1['random_cfm']` sampled from `pi_eb_random`. Metric space for selection: standardized PC-20 chord length. Display space: PHATE background only. Output: `figures/ch04/fig4_1_random_representative_pairs.png`.

In [ ]:
phate_display_limits = phate_limits(X0p_eb, X1p_eb)
random_rep_pair_path, random_rep_pairs = plot_representative_pairs(
    pair_stats_exp1["random_cfm"],
    X0p_eb,
    X1p_eb,
    filename="fig4_1_random_representative_pairs.png",
    title="Random coupling representative endpoint pairs",
    color=PALETTE["random"],
    limits=phate_display_limits,
)

### 1E. OT representative endpoint pairs
Claim: PC-20 OT coupling selects shorter and more locally coherent endpoint pairs than independent coupling. Data source: `pair_stats_exp1['ot_cfm']` sampled from `pi_eb_ot`. Metric space for selection: standardized PC-20 chord length. Display space: PHATE background only. Output: `figures/ch04/fig4_1_ot_representative_pairs.png`.

In [ ]:
ot_rep_pair_path, ot_rep_pairs = plot_representative_pairs(
    pair_stats_exp1["ot_cfm"],
    X0p_eb,
    X1p_eb,
    filename="fig4_1_ot_representative_pairs.png",
    title="OT coupling representative endpoint pairs",
    color=PALETTE["ot"],
    limits=phate_display_limits,
)

### 1F. Random vs OT chord-length distribution
Claim: OT-CFM uses the same standardized PC-20 cost space to shorten endpoint chords relative to independent coupling. Data sources: `exp1_metrics.csv` and `exp1_pair_level_statistics.csv` generated in this notebook. Metric space: standardized PC-20. Output: `figures/ch04/fig4_1_random_vs_ot_chord_length_distribution.png`.

In [ ]:
chord_distribution_path = plot_chord_length_ecdf(
    pair_stats_exp1,
    exp1_metrics,
    filename="fig4_1_random_vs_ot_chord_length_distribution.png",
)

### 1G. Random CFM representative rollouts
Claim: learned Random CFM ODE rollouts should be shown as a few representative examples, not as a global PHATE line cloud. Data source: `exp1_paths['random_cfm']`. Selection uses matched source cells from OT-CFM displacement quantiles for fair comparison. Metric space for model and rollout: standardized PC-20. Display space: PHATE only. Output: `figures/ch04/fig4_2_random_cfm_representative_rollouts.png`.

In [ ]:
rollout_representative_idx, rollout_representative_table = rollout_representative_indices(exp1_paths["ot_cfm"])
random_rollout_path = plot_representative_rollouts(
    exp1_paths["random_cfm"],
    rollout_representative_idx,
    X0p_eb,
    X1p_eb,
    pc_to_phate,
    filename="fig4_2_random_cfm_representative_rollouts.png",
    title="Random CFM representative rollouts",
    color=PALETTE["random"],
    limits=phate_display_limits,
)
display(rollout_representative_table.round(4))

### 1H. OT-CFM representative rollouts
Claim: OT-CFM rollouts should be compared on the same source examples as Random CFM, with generated endpoints marked clearly. Data source: `exp1_paths['ot_cfm']`. Metric space for model and rollout: standardized PC-20. Display space: PHATE only. Output: `figures/ch04/fig4_2_ot_cfm_representative_rollouts.png`.

In [ ]:
ot_rollout_path = plot_representative_rollouts(
    exp1_paths["ot_cfm"],
    rollout_representative_idx,
    X0p_eb,
    X1p_eb,
    pc_to_phate,
    filename="fig4_2_ot_cfm_representative_rollouts.png",
    title="OT-CFM representative rollouts",
    color=PALETTE["ot"],
    limits=phate_display_limits,
)

### 1I. Local velocity examples
Claim: local arrows are only representative visual examples; they are not the main evidence for endpoint fit or reflow straightness. Data source: Random CFM and OT-CFM rollout trajectories. Metric/training space: standardized PC-20. Display space: PHATE only, with arrows scaled for readability. Output: `figures/ch04/fig4_2_local_velocity_examples.png`.

In [ ]:
observed_phate = np.vstack([X0p_eb, X1p_eb])
local_velocity_idx = observed_near_velocity_indices(exp1_paths["ot_cfm"], pc_to_phate, observed_phate, n_arrows=6, seed=DEFAULT_SEED + 19)
local_velocity_path = plot_local_velocity_examples(
    {"random_cfm": exp1_paths["random_cfm"], "ot_cfm": exp1_paths["ot_cfm"]},
    local_velocity_idx,
    X0p_eb,
    X1p_eb,
    pc_to_phate,
    filename="fig4_2_local_velocity_examples.png",
    limits=phate_display_limits,
)
print("Local-arrow source indices:", local_velocity_idx.tolist())

## Exp 2. Sinkhorn Epsilon + Minibatch Ablation

This section isolates coupling-construction sensitivity while keeping the EB bridge and PC-20 preprocessing fixed. Part A varies Sinkhorn epsilon on the median-normalized PC-20 cost. Part B repeatedly recomputes Sinkhorn plans on source/target subsets to estimate minibatch-coupling variability.

Paper-facing outputs from Exp 2:
- `outputs/ch04/table4_A_sinkhorn_epsilon_ablation.csv`
- `outputs/ch04/tableA_4_1_minibatch_ablation.csv`
- `figures/ch04/fig4_2b_epsilon_ablation_pairs.png`

Raw epsilon plans and minibatch replicate rows are cache artifacts under `outputs/ch04/cache`.


### 2A. Vary Sinkhorn epsilon
This ablation changes only the entropic regularization on the same median-normalized PC-20 cost matrix.


In [ ]:
eps_rows = []
pi_by_eps = {}
for eps in EPSILON_GRID:
    pi, info = sinkhorn_plan(C_eb_norm, epsilon=eps, return_info=True)
    pi_by_eps[float(eps)] = pi
    idx0, idx1 = sample_from_plan(pi, 4096, seed=100 + int(eps * 1000))
    diag = coupling_diagnostics(pi, C_eb_norm)
    eps_rows.append({
        "epsilon": float(eps),
        "expected_cost_normalized": float(diag["expected_cost"]),
        "cost_scale": float(C_eb_scale),
        "entropy": float(diag["entropy"]),
        "effective_support": float(diag["effective_support"]),
        "sinkhorn_converged": bool(info["sinkhorn_converged"]),
        "sinkhorn_n_iter": int(info["n_iter"]),
        "row_l1_error": float(info["row_l1_error"]),
        "col_l1_error": float(info["col_l1_error"]),
        "warning_message": info.get("warning_message", ""),
        "mean_pair_distance_pc20": float(np.linalg.norm(X1_eb[idx1] - X0_eb[idx0], axis=1).mean()),
        "mean_pair_distance_phate_display_only": float(np.linalg.norm(X1p_eb[idx1] - X0p_eb[idx0], axis=1).mean()),
        "cost_space": "standardized PC-20 median-normalized",
        "phate_distance_role": "display diagnostic only",
    })


In [ ]:
eps_table = pd.DataFrame(eps_rows)
save_csv(OUT_DIR / "table4_A_sinkhorn_epsilon_ablation.csv", eps_table)
save_npz(CACHE_DIR / "exp2_epsilon_plans.npz", **{f"pi_eps_{str(k).replace('.', '_')}": v for k, v in pi_by_eps.items()})
eps_table


In [ ]:
# Epsilon pair visualization.
panels = []
for eps in [0.01, 0.05, 0.1, 0.5]:
    pi = pi_by_eps[float(eps)]
    idx0, idx1 = sample_from_plan(pi, 900, seed=120 + int(eps * 1000))
    panels.append({"title": f"epsilon={eps:g}", "idx0": idx0, "idx1": idx1, "color": PALETTE["ot"], "seed": 2})
plot_pair_panels(
    X0p_eb, X1p_eb, panels,
    "fig4_2b_epsilon_ablation_pairs.png",
    "Sinkhorn epsilon changes PC-20 coupling; displayed in PHATE only",
)

display_figure("fig4_2b_epsilon_ablation_pairs.png")


### 2B. Estimate minibatch coupling variability
The full plan is compared with repeated subset plans while preserving the same artifact names for the summary and raw replicate rows.


In [ ]:
minibatch_rows = []
mb_sizes = [64, 128, 256, 512, "full"]
rng = np.random.default_rng(202)
for mb in mb_sizes:
    repeats = 1 if mb == "full" else 10
    for rep in range(repeats):
        if mb == "full":
            x0_sub, x1_sub = X0_eb, X1_eb
        else:
            i0 = np.sort(rng.choice(len(X0_eb), size=min(int(mb), len(X0_eb)), replace=False))
            i1 = np.sort(rng.choice(len(X1_eb), size=min(int(mb), len(X1_eb)), replace=False))
            x0_sub, x1_sub = X0_eb[i0], X1_eb[i1]
        C_sub, scale_sub = compute_cost_matrix(x0_sub, x1_sub, normalize=True)
        pi_sub, info_sub = sinkhorn_plan(C_sub, epsilon=SINKHORN_EPSILON, return_info=True)
        diag = coupling_diagnostics(pi_sub, C_sub)
        minibatch_rows.append({
            "minibatch_size": str(mb),
            "repeat": int(rep),
            "n_source": int(len(x0_sub)),
            "n_target": int(len(x1_sub)),
            "expected_cost_normalized": float(diag["expected_cost"]),
            "effective_support": float(diag["effective_support"]),
            "entropy": float(diag["entropy"]),
            "sinkhorn_converged": bool(info_sub["sinkhorn_converged"]),
            "cost_scale": float(scale_sub),
        })
minibatch_raw = pd.DataFrame(minibatch_rows)


In [ ]:
minibatch_summary = minibatch_raw.groupby("minibatch_size", sort=False).agg(
    repeats=("repeat", "count"),
    expected_cost_mean=("expected_cost_normalized", "mean"),
    expected_cost_std=("expected_cost_normalized", "std"),
    effective_support_mean=("effective_support", "mean"),
    effective_support_std=("effective_support", "std"),
).reset_index()
save_csv(OUT_DIR / "tableA_4_1_minibatch_ablation.csv", minibatch_summary)
save_csv(CACHE_DIR / "exp2_minibatch_ablation_raw.csv", minibatch_raw)
minibatch_summary


## Exp 3. Rectified Flow

This experiment uses the Exp 1 OT-CFM as the base model, then trains two reflow rounds from model-induced endpoints. Endpoint metrics are still computed against the real EB target distribution in standardized PC-20, but the main claim for reflow is path straightness and coarse-step solver behavior.

Paper-facing outputs from Exp 3:
- `outputs/ch04/table4_1_reflow_ablation.csv`
- `outputs/ch04/cache/exp3_nfe_sensitivity.csv`
- new small figures under `figures/ch04/fig4_3_*`

PHATE is used only as a display projection for the sparse representative trajectory examples.

### 3A. Train reflow rounds from induced endpoints
The reflow targets come from model rollouts and are cached before each diagonal-coupling training round.

In [ ]:
def train_reflow_round(name: str, base_model, X0, steps: int = TRAINING_STEPS, seed: int = DEFAULT_SEED):
    endpoint_path = CACHE_DIR / f"{name}_induced_endpoint.npz"
    if endpoint_path.exists():
        Z = load_npz(endpoint_path)["endpoint"]
    else:
        Z = rollout_euler(base_model, X0, nfe=DEFAULT_NFE)
        save_npz(endpoint_path, endpoint=Z)
    pi_diag = np.eye(len(X0), len(Z), dtype=float)
    pi_diag /= pi_diag.sum()
    model, hist = train_or_load_model(name, X0, Z, pi_diag, steps=steps, seed=seed)
    return model, Z, hist


In [ ]:
reflow1_model, Z1_1, reflow1_hist = train_reflow_round("exp3_reflow1", ot_model, X0_eb, seed=43)
reflow2_model, Z1_2, reflow2_hist = train_reflow_round("exp3_reflow2", reflow1_model, X0_eb, seed=44)


### 3B. Evaluate reflow geometry and NFE sensitivity
Endpoint metrics remain anchored to the real EB target distribution in standardized PC-20.

In [ ]:
models_exp3 = {
    "random_cfm": random_model,
    "ot_cfm": ot_model,
    "reflow_1": reflow1_model,
    "reflow_2": reflow2_model,
}
traj_exp3 = {}
rows = []
for name, model in models_exp3.items():
    z, traj, times = trajectory_rollout(model, X0_eb, nfe=DEFAULT_NFE, return_path=True)
    traj_exp3[name] = traj
    rows.append({
        "method": name,
        **evaluate_endpoint(z, X1_eb),
        "straightness_action_S": straightness_action_S(traj, times),
        "tortuosity_straightness": tortuosity_straightness(traj),
        "path_length": path_length(traj),
        "path_energy": path_energy(traj, times),
        "coarse_step_error_nfe4_vs_nfe64": coarse_step_error(model, X0_eb, nfe_coarse=4, nfe_fine=64),
        "endpoint_reference": "real EB target distribution in standardized PC-20",
    })
reflow_table = pd.DataFrame(rows)
save_csv(OUT_DIR / "table4_1_reflow_ablation.csv", reflow_table)

reflow_table


In [ ]:
nfe_cache_path = CACHE_DIR / "exp3_nfe_sensitivity.csv"
if nfe_cache_path.exists():
    nfe_table = pd.read_csv(nfe_cache_path)
else:
    nfe_rows = []
    for name, model in models_exp3.items():
        for nfe in NFE_GRID:
            z = rollout_euler(model, X0_eb, nfe=nfe)
            nfe_rows.append({"method": name, "nfe": int(nfe), **evaluate_endpoint(z, X1_eb)})
    nfe_table = pd.DataFrame(nfe_rows)
    save_csv(nfe_cache_path, nfe_table)
nfe_table


### 3C. Reflow representative trajectories
Claim: reflow trajectory displays should be sparse examples using the same source cells across methods; straightness is supported by the statistics below. Data source: `traj_exp3` generated from OT-CFM, Reflow 1, and Reflow 2. Metric/training space: standardized PC-20. Display space: PHATE only. Output: `figures/ch04/fig4_3_reflow_representative_trajectories.png`.

In [ ]:
if "rollout_representative_idx" not in globals():
    rollout_representative_idx, rollout_representative_table = rollout_representative_indices(traj_exp3["ot_cfm"])
reflow_rep_path = plot_reflow_representative_trajectories(
    traj_exp3,
    rollout_representative_idx,
    X0p_eb,
    X1p_eb,
    pc_to_phate,
    filename="fig4_3_reflow_representative_trajectories.png",
    limits=phate_display_limits,
)

### 3D. Reflow straightness and path-geometry summary
Claim: the main reflow evidence is reduced path straightness action and lower path energy, not a denser PHATE trajectory cloud. Data source: `table4_1_reflow_ablation.csv` / current `reflow_table`. Metric space: standardized PC-20 trajectories. Output: `figures/ch04/fig4_3_reflow_straightness_summary.png`.

In [ ]:
reflow_summary_table = reflow_table.rename(columns={"path_energy": "path_energy_action"}).copy()
straightness_metric_specs = [
    ("straightness_action_S", "Straightness action S", "lower is straighter"),
    ("tortuosity_straightness", "Tortuosity minus 1", "lower is straighter"),
    ("path_length", "Path length", "standardized PC-20"),
    ("path_energy_action", "Path energy/action", "standardized PC-20"),
]
straightness_summary_path = plot_metric_bar_grid(
    reflow_summary_table,
    methods=["random_cfm", "ot_cfm", "reflow_1", "reflow_2"],
    metric_specs=straightness_metric_specs,
    filename="fig4_3_reflow_straightness_summary.png",
    log_metrics={"straightness_action_S", "tortuosity_straightness"},
)

### 3E. Coarse-step solver behavior
Claim: reflow improves coarse-step solver behavior, measured as mean endpoint deviation between NFE=4 and NFE=64 in standardized PC-20. Data source: `table4_1_reflow_ablation.csv`; `exp3_nfe_sensitivity.csv` is retained as the multi-NFE endpoint-fit cache. Output: `figures/ch04/fig4_3_reflow_nfe_error_summary.png`.

In [ ]:
nfe_error_metric_specs = [("coarse_step_error_nfe4_vs_nfe64", "NFE 4 vs NFE 64 endpoint deviation", "mean L2 in standardized PC-20")]
nfe_error_summary_path = plot_metric_bar_grid(
    reflow_table,
    methods=["random_cfm", "ot_cfm", "reflow_1", "reflow_2"],
    metric_specs=nfe_error_metric_specs,
    filename="fig4_3_reflow_nfe_error_summary.png",
)
display(nfe_table.loc[nfe_table["nfe"].isin([4, 64])].sort_values(["method", "nfe"]).round(4))

## Exp 4. Coupling Diagnostic Table

The final experiment summarizes endpoint, path-geometry, off-manifold, and coarse-step diagnostics for `random_cfm`, `ot_cfm`, `reflow_1`, and `reflow_2`. The off-manifold reference is all observed EB snapshots in standardized PC-20 when available.

Paper-facing output from Exp 4:
- `outputs/ch04/table4_1_path_geometry_diagnostics.csv`

The section also refreshes `figures/ch04/fig4_5_random_vs_ot_projected_trajectories.png` and writes progress state to `outputs/ch04/cache/exp4_path_diagnostics_progress.json` while the table is being assembled.


### 4A. Build the diagnostic table incrementally
The progress JSON is refreshed after each method so interrupted long runs leave an auditable state.


In [ ]:
import gc

path_diag_path = OUT_DIR / "table4_1_path_geometry_diagnostics.csv"
required_path_diag_methods = {"random_cfm", "ot_cfm", "reflow_1", "reflow_2"}
required_path_diag_columns = {
    "method",
    "endpoint_mmd",
    "sliced_w2",
    "path_length",
    "path_energy_action",
    "straightness_action_S",
    "tortuosity_straightness",
    "off_manifold_knn_k15",
    "coarse_step_error_nfe4_vs_nfe64",
    "training_space",
    "endpoint_metric_space",
}
path_diag_table = None
if path_diag_path.exists():
    cached_path_diag = pd.read_csv(path_diag_path)
    has_methods = required_path_diag_methods.issubset(set(cached_path_diag.get("method", [])))
    has_columns = required_path_diag_columns.issubset(set(cached_path_diag.columns))
    if has_methods and has_columns:
        path_diag_table = cached_path_diag

if path_diag_table is None:
    path_diag_rows = []
    exp4_progress_path = CACHE_DIR / "exp4_path_diagnostics_progress.json"
    for name, model in models_exp3.items():
        save_json(exp4_progress_path, {"stage": "starting", "method": name, "completed_methods": [r["method"] for r in path_diag_rows]})
        z, traj, times = trajectory_rollout(model, X0_eb, nfe=DEFAULT_NFE, return_path=True)
        row = {
            "method": name,
            **evaluate_endpoint(z, X1_eb),
            "path_length": path_length(traj),
            "path_energy_action": path_energy(traj, times),
            "straightness_action_S": straightness_action_S(traj, times),
            "tortuosity_straightness": tortuosity_straightness(traj),
            "off_manifold_knn_k15": off_manifold_knn(traj, off_manifold_reference_pc, k=15),
            "coarse_step_error_nfe4_vs_nfe64": coarse_step_error(model, X0_eb, nfe_coarse=4, nfe_fine=64),
            "off_manifold_reference": off_manifold_reference_note,
            "training_space": "standardized PC-20",
            "endpoint_metric_space": "standardized PC-20 real target",
        }
        path_diag_rows.append(row)
        save_csv(path_diag_path, pd.DataFrame(path_diag_rows))
        save_json(exp4_progress_path, {"stage": "completed", "method": name, "completed_methods": [r["method"] for r in path_diag_rows]})
        del z, traj
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    path_diag_table = pd.DataFrame(path_diag_rows)
else:
    print(f"Loaded cached path diagnostics from {path_diag_path}")


In [ ]:
path_diag_table


### 4B. Random vs OT endpoint-fit and path-energy summary
Claim: OT-CFM improves endpoint fit and lowers path energy relative to Random CFM; this quantitative evidence is stronger than a global PHATE trajectory cloud. Data source: `table4_1_path_geometry_diagnostics.csv` / current `path_diag_table`. Metric space: standardized PC-20. Output: `figures/ch04/fig4_2_random_vs_ot_endpoint_fit_summary.png`.

In [ ]:
endpoint_fit_metric_specs = [
    ("endpoint_mmd", "Endpoint MMD", "standardized PC-20"),
    ("sliced_w2", "Sliced W2", "standardized PC-20"),
    ("path_length", "Path length", "standardized PC-20"),
    ("path_energy_action", "Path energy/action", "standardized PC-20"),
]
endpoint_fit_summary_path = plot_metric_bar_grid(
    path_diag_table,
    methods=["random_cfm", "ot_cfm"],
    metric_specs=endpoint_fit_metric_specs,
    filename="fig4_2_random_vs_ot_endpoint_fit_summary.png",
)

## Artifact Manifest
The final cells record the run configuration, the new small-figure set, legacy provenance figures that remain available, and the CSV/cache dependencies used by the new figures. The newly generated figures are saved as separate PNG files for manual PPT layout.


In [ ]:
legacy_provenance_figures = [
    "fig4_1_independent_coupling_paths.png",
    "fig4_2_random_vs_ot_pairs.png",
    "fig4_1_supp_pair_statistics.png",
    "fig4_2b_epsilon_ablation_pairs.png",
    "fig4_4_reflow_trajectories.png",
    "fig4_5_random_vs_ot_projected_trajectories.png",
    "fig4_4b_nfe_endpoint_error.png",
]
new_small_figures = [
    "fig4_1_random_representative_pairs.png",
    "fig4_1_ot_representative_pairs.png",
    "fig4_1_random_vs_ot_chord_length_distribution.png",
    "fig4_2_random_cfm_representative_rollouts.png",
    "fig4_2_ot_cfm_representative_rollouts.png",
    "fig4_2_local_velocity_examples.png",
    "fig4_3_reflow_representative_trajectories.png",
    "fig4_3_reflow_straightness_summary.png",
    "fig4_3_reflow_nfe_error_summary.png",
    "fig4_2_random_vs_ot_endpoint_fit_summary.png",
]
expected_figures = legacy_provenance_figures + new_small_figures
expected_tables = [
    "eb_data_summary.json",
    "exp1_metrics.csv",
    "table4_A_sinkhorn_epsilon_ablation.csv",
    "tableA_4_1_minibatch_ablation.csv",
    "table4_1_reflow_ablation.csv",
    "table4_1_path_geometry_diagnostics.csv",
]
figure_dependency_files = [
    CACHE_DIR / "exp1_eb_couplings.npz",
    CACHE_DIR / "exp1_pair_level_statistics.csv",
    CACHE_DIR / "exp1_random_cfm_rollout.npz",
    CACHE_DIR / "exp1_ot_cfm_rollout.npz",
    CACHE_DIR / "exp3_reflow1_induced_endpoint.npz",
    CACHE_DIR / "exp3_reflow2_induced_endpoint.npz",
    CACHE_DIR / "exp3_nfe_sensitivity.csv",
    OUT_DIR / "exp1_metrics.csv",
    OUT_DIR / "table4_1_reflow_ablation.csv",
    OUT_DIR / "table4_1_path_geometry_diagnostics.csv",
]


In [ ]:
run_config = {
    "SMOKE_MODE": bool(SMOKE_MODE),
    "TRAINING_STEPS": int(TRAINING_STEPS),
    "BATCH_SIZE": int(BATCH_SIZE),
    "DEFAULT_NFE": int(DEFAULT_NFE),
    "DEVICE": str(DEVICE),
}
save_json(OUT_DIR / "run_config_04_1_coupling_geometry.json", run_config)


In [ ]:
# Manifest artifact names are intentionally listed in this cell for split-notebook tests and auditability.
manifest_expected_figures = [
    "fig4_1_independent_coupling_paths.png",
    "fig4_2_random_vs_ot_pairs.png",
    "fig4_1_supp_pair_statistics.png",
    "fig4_2b_epsilon_ablation_pairs.png",
    "fig4_4_reflow_trajectories.png",
    "fig4_5_random_vs_ot_projected_trajectories.png",
    "fig4_4b_nfe_endpoint_error.png",
    "fig4_1_random_representative_pairs.png",
    "fig4_1_ot_representative_pairs.png",
    "fig4_1_random_vs_ot_chord_length_distribution.png",
    "fig4_2_random_cfm_representative_rollouts.png",
    "fig4_2_ot_cfm_representative_rollouts.png",
    "fig4_2_local_velocity_examples.png",
    "fig4_3_reflow_representative_trajectories.png",
    "fig4_3_reflow_straightness_summary.png",
    "fig4_3_reflow_nfe_error_summary.png",
    "fig4_2_random_vs_ot_endpoint_fit_summary.png",
]
if manifest_expected_figures != expected_figures:
    raise ValueError("expected_figures and manifest_expected_figures diverged")

manifest_expected_tables = [
    "eb_data_summary.json",
    "exp1_metrics.csv",
    "table4_A_sinkhorn_epsilon_ablation.csv",
    "tableA_4_1_minibatch_ablation.csv",
    "table4_1_reflow_ablation.csv",
    "table4_1_path_geometry_diagnostics.csv",
]
if manifest_expected_tables != expected_tables:
    raise ValueError("expected_tables and manifest_expected_tables diverged")

manifest_rows = []
for key, value in run_config.items():
    manifest_rows.append({"artifact": f"RUN_CONFIG:{key}={value}", "kind": "run_config", "exists": True, "bytes": 0})
for name in expected_figures:
    p = FIG_DIR / name
    manifest_rows.append({"artifact": str(p.relative_to(PROJECT_ROOT)), "kind": "figure", "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else 0})
for name in expected_tables:
    p = OUT_DIR / name
    manifest_rows.append({"artifact": str(p.relative_to(PROJECT_ROOT)), "kind": "table_or_json", "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else 0})
for p in figure_dependency_files:
    manifest_rows.append({"artifact": str(p.relative_to(PROJECT_ROOT)), "kind": "dependency", "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else 0})
artifact_manifest = pd.DataFrame(manifest_rows)
save_csv(OUT_DIR / "artifact_manifest_04_1_coupling_geometry.csv", artifact_manifest)


In [ ]:
missing = artifact_manifest.loc[(artifact_manifest["kind"] != "run_config") & (~artifact_manifest["exists"])]
if not missing.empty:
    raise FileNotFoundError("Missing expected Chapter 4 split artifacts: " + ", ".join(missing["artifact"].tolist()))

new_figure_manifest = artifact_manifest[artifact_manifest["artifact"].isin([str((FIG_DIR / name).relative_to(PROJECT_ROOT)) for name in new_small_figures])].copy()
dependency_manifest = artifact_manifest[artifact_manifest["kind"] == "dependency"].copy()
print("New small figures")
display(new_figure_manifest[["artifact", "bytes"]])
print("Figure dependencies")
display(dependency_manifest[["artifact", "bytes"]])
print("Full artifact manifest")
display(artifact_manifest)
